In [1]:
# ── Cell 0: Setup (chạy đầu tiên) ──
import os

# Clone repo nếu chưa có
if not os.path.exists('/content/project'):
    !git clone https://github.com/TruongDuyLongPTIT/CTQW_PRO_METABOLITES_PRIORITIZING.git /content/project

# Add src vào Python path
import sys
sys.path.insert(0, '/content/project/src')

# Install dependencies nếu cần
!pip install -q torch scikit-learn networkx tqdm

Cloning into '/content/project'...
remote: Enumerating objects: 243, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 243 (delta 17), reused 24 (delta 10), pack-reused 203 (from 1)
Receiving objects: 100% (243/243), 1.15 MiB | 7.96 MiB/s, done.
Resolving deltas: 100% (137/137), done.


In [2]:
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print("Drive đã mount sẵn.")

Mounted at /content/drive


In [ ]:
!python /content/project/experiments/01_main_results.py

STEP 1 — Build graph
  Graph: 2788 nodes, 22439 edges
  G_pro: 2894 nodes (106 pathway), 31360 edges

STEP 2 — Build eval sets
  hmdb_to_recon: +0 IK, +331 name → 3286 total
  Extracting SMPDB metabolites...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:26<00:00, 1862.54it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  eval_set1 (HMDB+CTD): 158 diseases
  eval_set2 (MarkerDB): 21 diseases
  eval_set3 (SMPDB):    153 diseases

STEP 3 — Eigendecomposition
  Done.

[Table 1] RWR vs CTQW on G_cc...
RWR/HMDB+CTD: 100% 158/158 [01:28<00:00,  1.78it/s]
CTQW/HMDB+CTD: 100% 158/158 [03:38<00:00,  1.38s/it]
  HMDB+CTD: 5.1 min
RWR/MarkerDB: 100% 21/21 [00:15<00:00,  1.32it/s]
CTQW/MarkerDB: 100% 21/21 [00:39<00:00,  1.88s/it]
  MarkerDB: 0.9 min
RWR/SMPDB: 100% 153/153 [01:03<00:00,  2.41it/s]
CTQW/SMPDB: 100% 153/153 [02:45<00:00,  1.08s/it]
  SMPDB: 3.8 min

[Table 2] PROFANCY vs CTQW-PRO on G_pro...
PROFANCY/HMDB+CTD: 100% 158/158 [01:40<00:00,  1.58it

In [ ]:
!python /content/project/experiments/02_ablation_graph.py

Building graphs...
  G_pro: 2894 → clean: 2851 nodes
Building eval sets...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:30<00:00, 1587.66it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
Eigendecomposition...

Running ablation...
PROF_o/HMDB+CTD: 100% 158/158 [01:38<00:00,  1.61it/s]
CTQW_o/HMDB+CTD: 100% 158/158 [04:20<00:00,  1.65s/it]
PROF_c/HMDB+CTD: 100% 158/158 [01:39<00:00,  1.59it/s]
CTQW_c/HMDB+CTD: 100% 158/158 [04:55<00:00,  1.87s/it]
  HMDB+CTD: 12.6 min
PROF_o/MarkerDB: 100% 21/21 [00:19<00:00,  1.09it/s]
CTQW_o/MarkerDB: 100% 21/21 [00:46<00:00,  2.22s/it]
PROF_c/MarkerDB: 100% 21/21 [00:18<00:00,  1.13it/s]
CTQW_c/MarkerDB: 100% 21/21 [00:53<00:00,  2.55s/it]
  MarkerDB: 2.3 min
PROF_o/SMPDB: 100% 153/153 [01:11<00:00,  2.13it/s]
CTQW_o/SMPDB: 100% 153/153 [03:13<00:00,  1.26s/it]
PROF_c/SMPDB: 100% 153/153 [01:14<00:00,  2.05it/s]
CTQW_c/SMPDB: 100% 153/153 [03:43<00:00,  1.46s/it]
  SMPDB: 9.4 min

ABLATION: Original vs Clean G_p

In [ ]:
!python /content/project/experiments/03_negative_results.py

Setup...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:26<00:00, 1867.61it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158 diseases
  SMPDB:    153 diseases
  Device: cpu

EXP 1: Self-loop leakage analysis (GPU-accelerated)
  (A) Baseline:     no self-loops (γ=0)
  (B) Leaked:       H = A_pro + 10.0·diag(ALL disease mets in G_pro)
  (C) Leakage-free: H = A_pro + 10.0·diag(seed mets only, per fold)
  Semantics of B: test_met receives self-loop → score artificially boosted
  Dataset: full SMPDB (153 diseases)
  NOTE: uses torch.linalg.eigh (symmetric H) for all 3 conditions
  [ 10/153] elapsed=17.4m  ETA=249.5m | MRR  A=0.2561  B=0.3759  C=0.2379 | C_eigh=139.8s
  [ 20/153] elapsed=38.8m  ETA=257.7m | MRR  A=0.2388  B=0.3278  C=0.2275 | C_eigh=69.1s
  [ 30/153] elapsed=54.9m  ETA=225.3m | MRR  A=0.2460  B=0.3280  C=0.2335 | C_eigh=87.3s
  [ 40/153] elapsed=73.5m  ETA=207.7m | MRR  A=0.2318  B=0.3154  C=0.2192 | C_eigh=83.4s
  [ 50/153

In [ ]:
!python /content/project/experiments/06_biological_interpretability.py

06 — Biological Interpretability Analysis
Setup...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:28<00:00, 1702.96it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215

Building NH-CTQW-PRO (γ=22.0, t=0.1)...
  Done in 52.0s

DISEASE: Lesch-Nyhan Syndrome (LNS)  [LNS]

Seeds (n=24) — known disease metabolites:
    #  Name                                                         HMDB  HMDB link
  -----------------------------------------------------------------------------------------------
    1. (S)-2-[5-Amino-1-(5-phospho-D-ribosyl)imidazole-4-carboxamido]succinate    HMDB0000797  https://hmdb.ca/metabolites/HMDB0000797
    2. Adenine                                               HMDB0000034  https://hmdb.ca/metabolites/HMDB0000034
    3. Adenosine                                             HMDB0004402  https://hmdb.ca/metabolites/HMDB0004402
    4. 5-Amino-1-(5-Phospho-D-ribosyl)imidazole-4-carboxamide    HMDB0001517  https://hmdb.ca/metabolites/HM

### **-----------Kiểm định phụ-----------------**

In [4]:
"""
verify_cross_graph_augmentation_cell.py — Paste nguyên cell vào main_notebook, chạy trực tiếp.

MỤC ĐÍCH: kiểm định thống kê CHÉO ĐỒ THỊ (cùng phương pháp, Gcc vs Gpro) cho insight
mới của 5.2: "augment pathway node giúp CTQW nhiều hơn RWR". Bảng 1 (RWR vs CTQW/Gcc)
và Bảng 2 (PROFANCY vs CTQW-PRO/Gpro) chỉ kiểm định TRONG-CÙNG-ĐỒ-THỊ — chưa có test
nào so RWR-trên-Gcc với PROFANCY-trên-Gpro (=RWR-trên-Gpro), hay CTQW-trên-Gcc với
CTQW-PRO-trên-Gpro. Cell này dùng ĐÚNG cùng hàm wilcoxon_table() (paired theo bệnh,
Bonferroni-corrected, exact như cách Bảng 1/2/3 đã được kiểm định trong 01_main_results.py)
để lấp khoảng trống này, cho 2 cặp so sánh:

  (A) Cổ điển: PROFANCY (RWR/Gpro)  vs  RWR (Gcc)   — delta = Gpro - Gcc
  (B) Lượng tử: CTQW-PRO (CTQW/Gpro) vs  CTQW (Gcc)  — delta = Gpro - Gcc

trên cả 3 bộ dữ liệu (HMDB+CTD, MarkerDB, SMPDB).
"""
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd

from config import RESULTS_DIR, CACHE_DIR, T_FIXED
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups, build_cofactors_set,
                       build_eval_set1, build_eval_set2, build_eval_set3)
from methods import run_rwr, make_profancy, make_ctqw_pro, make_ctqw_gcc
from evaluation import run_loo_eval, wilcoxon_table

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('='*60)
print('STEP 1 — Build graph (giống hệt 01_main_results.py)')
recon_data   = parse_recon3d()
G_cc, _, N, node_idx, A_cc, _ = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

deg_pro_safe = np.where(deg_pro > 0, deg_pro, 1.0)
P_pro        = A_pro / deg_pro_safe[:, None]
deg_cc_safe  = np.where(A_cc.sum(1) > 0, A_cc.sum(1), 1.0)
P_cc         = A_cc / deg_cc_safe[:, None]

print(f'  Gcc: {N} nodes | Gpro: {N_PRO} nodes')

print('\nSTEP 2 — Build eval sets')
hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)

eval_set1, disease_canonical = build_eval_set1(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon, node_idx, COFACTORS)
eval_set2 = build_eval_set2(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon,
    node_idx, COFACTORS, disease_canonical)
eval_set3 = build_eval_set3(
    hmdb_metabolites, hmdb_to_recon, node_idx, COFACTORS)
print(f'  HMDB+CTD: {len(eval_set1)} | MarkerDB: {len(eval_set2)} | SMPDB: {len(eval_set3)}')

print('\nSTEP 3 — Eigendecomposition')
Apro_eigvals, Apro_eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')
Acc_eigvals, Acc_eigvecs   = compute_eigendecomp(A_cc,  CACHE_DIR / 'gcc_eigdecomp.npz')

print('\nSTEP 4 — Chạy LOO cho 4 phương pháp x 3 bộ dữ liệu')
_rwr_fn      = lambda seeds: run_rwr(seeds, P_cc, node_idx, N)
_ctqw_gcc    = make_ctqw_gcc(Acc_eigvals, Acc_eigvecs, N)
_ctqw_gcc_fn = lambda seeds: _ctqw_gcc(seeds, node_idx)
run_profancy = make_profancy(P_pro, idx_pro, node_idx, N, N_PRO)
run_ctqw_pro = make_ctqw_pro(Apro_eigvals, Apro_eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst)
_ctqw_pro_fn = lambda seeds: run_ctqw_pro(seeds, [T_FIXED])[T_FIXED]

dfs = {}
for label, dset in [('HMDB+CTD', eval_set1), ('MarkerDB', eval_set2), ('SMPDB', eval_set3)]:
    t0 = time.time()
    df_rwr  = run_loo_eval(dset, _rwr_fn,       node_idx, N, label=f'RWR/{label}')
    df_prof = run_loo_eval(dset, run_profancy,  node_idx, N, label=f'PROFANCY/{label}')
    df_ctqw = run_loo_eval(dset, _ctqw_gcc_fn,  node_idx, N, label=f'CTQW/{label}')
    df_cpro = run_loo_eval(dset, _ctqw_pro_fn,  node_idx, N, label=f'CTQW-PRO/{label}')
    dfs[label] = {'RWR': df_rwr, 'PROFANCY': df_prof, 'CTQW': df_ctqw, 'CTQW-PRO': df_cpro}
    print(f'  {label}: {(time.time()-t0)/60:.1f} min')

print('\n' + '='*72)
print('KIỂM ĐỊNH CHÉO ĐỒ THỊ (A) — CỔ ĐIỂN: PROFANCY (Gpro) vs RWR (Gcc)')
print('  delta = PROFANCY - RWR  (dương = Gpro tốt hơn Gcc)')
print('='*72)
classical_rows = []
for label in ['HMDB+CTD', 'MarkerDB', 'SMPDB']:
    df_wx = wilcoxon_table(dfs[label]['PROFANCY'], dfs[label]['RWR'], label,
                           method_a='PROFANCY', method_b='RWR')
    if df_wx is not None:
        df_wx['comparison'] = 'classical (RWR Gcc -> PROFANCY Gpro)'
        classical_rows.append(df_wx)

print('\n' + '='*72)
print('KIỂM ĐỊNH CHÉO ĐỒ THỊ (B) — LƯỢNG TỬ: CTQW-PRO (Gpro) vs CTQW (Gcc)')
print('  delta = CTQW-PRO - CTQW  (dương = Gpro tốt hơn Gcc)')
print('='*72)
quantum_rows = []
for label in ['HMDB+CTD', 'MarkerDB', 'SMPDB']:
    df_wx = wilcoxon_table(dfs[label]['CTQW-PRO'], dfs[label]['CTQW'], label,
                           method_a='CTQW-PRO', method_b='CTQW')
    if df_wx is not None:
        df_wx['comparison'] = 'quantum (CTQW Gcc -> CTQW-PRO Gpro)'
        quantum_rows.append(df_wx)

if classical_rows or quantum_rows:
    out = pd.concat(classical_rows + quantum_rows, ignore_index=True)
    out_path = RESULTS_DIR / 'cross_graph_augmentation_wilcoxon.csv'
    out.to_csv(out_path, index=False)
    print(f'\nSaved: {out_path}')

print('\nDone. Dán TOÀN BỘ output (cả 2 bảng kiểm định) cho tôi để đưa vào 5.2 với '
      'p-value thật thay vì chỉ point estimate.')

STEP 1 — Build graph (giống hệt 01_main_results.py)
  Gcc: 2788 nodes | Gpro: 2894 nodes

STEP 2 — Build eval sets
  Extracting SMPDB metabolites...


Extract SMPDB:   0%|          | 0/48687 [00:00<?, ?it/s]

  Disease pathways: 20248


SMPDB files:   0%|          | 0/48687 [00:00<?, ?it/s]

  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158 | MarkerDB: 21 | SMPDB: 153

STEP 3 — Eigendecomposition

STEP 4 — Chạy LOO cho 4 phương pháp x 3 bộ dữ liệu


...

PROFANCY/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]

CTQW/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]

CTQW-PRO/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]

  SMPDB: 16.5 min

KIỂM ĐỊNH CHÉO ĐỒ THỊ (A) — CỔ ĐIỂN: PROFANCY (Gpro) vs RWR (Gcc)
  delta = PROFANCY - RWR  (dương = Gpro tốt hơn Gcc)

--- HMDB+CTD (n=158) ---
  Metric                RWR         PROFANCY    Delta     p_bonf  Sig
  auc        0.7872+/-0.0997   0.8261+/-0.0794  +0.0389   1.64e-26  ***
  mrr        0.0158+/-0.0195   0.0161+/-0.0200  +0.0003  4.632e-05  ***
  r@5        0.0080+/-0.0348   0.0059+/-0.0322  -0.0021     0.4073  ns
  r@10       0.0240+/-0.0753   0.0251+/-0.0830  +0.0011          1  ns
  r@20       0.0798+/-0.1520   0.0879+/-0.1582  +0.0081    0.07414  .
  r@50       0.2186+/-0.2409   0.2189+/-0.2397  +0.0004          1  ns

--- MarkerDB (n=21) ---
  Metric                RWR         PROFANCY    Delta     p_bonf  Sig
  auc        0.8541+/-0.0973   0.8837+/-0.0740  +0.0296  0.0008408  ***
  mrr        0.0279+/-0.0255   0.0287+/-0.0255  +0.0007   0.008245  **
  r@5        0.0115+/-0.0293   0.0115+/-0.0293  +0.0000        nan  
  r@10       0.0386+/-0.0620   0

/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


In [6]:
"""
verify_dephasing_performance_cell.py — Paste nguyên cell vào main_notebook, chạy trực tiếp.

MỤC ĐÍCH: thực nghiệm dephasing gốc (03_negative_results.py, EXP 3) CHỈ đo khoảng cách
L2 giữa phân bố xác suất và phân bố mất pha hoàn toàn, trên ĐÚNG 1 bệnh — KHÔNG đo hiệu
suất xếp hạng (MRR/AUC/R@k) qua LOO. Cell này bổ sung phần còn thiếu, thu hẹp phạm vi để
chạy nhanh theo đúng yêu cầu:

  - Chỉ chạy trên 15 bệnh đang có delta_mrr (CTQW-PRO − PROFANCY, tức mức CTQW-PRO cải
    thiện so với PROFANCY, giống hệt Bảng 2) TỐT NHẤT của HMDB+CTD, và 15 bệnh tốt nhất
    tương tự của SMPDB — để xem chính những ca đang hưởng lợi nhiều nhất từ giao thoa
    lượng tử có mất lợi thế đó khi bị dephase hay không.
  - Chỉ so sánh 2 mức: sigma=0.0 (CTQW-PRO gốc, dùng làm sanity check) và sigma=5.0
    (gần như mất hoàn toàn tính mạch lạc pha, theo đúng grid đã dùng ở test L2-distance
    cũ) — vì cơ chế đã được chứng minh qua ablation + degree-correlation, giờ chỉ cần
    1 điểm dữ liệu hiệu suất thực nghiệm ở đầu và cuối thang đo.
  - N_MC_PER_FOLD = 10.

Cơ chế dephasing giữ nguyên công thức gốc: nhiễu Gauss độc lập trên PHA của từng
eigenmode (η_k ~ N(0, σ²)), lấy trung bình Monte Carlo N_MC_PER_FOLD lần mỗi lượt LOO.
"""
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd

from config import RESULTS_DIR, CACHE_DIR, T_FIXED, RANDOM_SEED
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups, build_cofactors_set,
                       build_eval_set1, build_eval_set3)
from methods import make_profancy, make_ctqw_pro
from evaluation import run_loo_eval, wilcoxon_table

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Cấu hình ──────────────────────────────────────────────────────────────
SIGMA_GRID    = [0.0, 5.0]   # 0.0 = CTQW-PRO gốc (sanity check), 5.0 = gần mất hết pha
N_MC_PER_FOLD = 10
N_TOP         = 15           # số bệnh "tốt nhất" lấy mỗi bộ dữ liệu

print('='*60)
print('STEP 1 — Build graph')
recon_data   = parse_recon3d()
G_cc, _, N, node_idx, A_cc, _ = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

deg_pro_safe = np.where(deg_pro > 0, deg_pro, 1.0)
P_pro        = A_pro / deg_pro_safe[:, None]

print('\nSTEP 2 — Build eval sets (chỉ HMDB+CTD và SMPDB)')
hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)

eval_set1, disease_canonical = build_eval_set1(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon, node_idx, COFACTORS)
eval_set3 = build_eval_set3(
    hmdb_metabolites, hmdb_to_recon, node_idx, COFACTORS)
print(f'  HMDB+CTD: {len(eval_set1)} bệnh | SMPDB: {len(eval_set3)} bệnh')

print('\nSTEP 3 — Eigendecomposition (Gpro)')
Apro_eigvals, Apro_eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

_ph_fixed = np.exp(-1j * Apro_eigvals * float(T_FIXED))
_rng      = np.random.default_rng(RANDOM_SEED)

run_profancy = make_profancy(P_pro, idx_pro, node_idx, N, N_PRO)
run_ctqw_pro = make_ctqw_pro(Apro_eigvals, Apro_eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst)
_ctqw_pro_fn = lambda seeds: run_ctqw_pro(seeds, [T_FIXED])[T_FIXED]


def make_ctqw_pro_dephased(sigma, n_mc=N_MC_PER_FOLD):
    """CTQW-PRO với nhiễu pha Gauss trên từng eigenmode. sigma=0 -> CTQW-PRO gốc."""
    def run_dephased(seed_nodes, _n=N):
        valid = [idx_pro[s] for s in seed_nodes if s in idx_pro]
        if not valid:
            return np.zeros(_n)
        psi0 = np.zeros(N_PRO, dtype=complex)
        psi0[valid] = 1.0 / np.sqrt(len(valid))
        c = Apro_eigvecs.conj().T @ psi0
        if sigma == 0.0:
            probs = np.abs(Apro_eigvecs @ (_ph_fixed * c))**2
        else:
            probs = np.zeros(N_PRO)
            for _ in range(n_mc):
                noise = _rng.normal(0.0, sigma, size=N_PRO)
                probs += np.abs(Apro_eigvecs @ ((_ph_fixed * np.exp(1j * noise)) * c))**2
            probs /= n_mc
        sc = np.zeros(_n)
        sc[_pro_dst] = probs[_pro_src]
        return sc
    return run_dephased


print('\nSTEP 4 — Tính delta_mrr (CTQW-PRO - PROFANCY) trên TOÀN BỘ mỗi bộ dữ liệu, '
      'chọn 15 bệnh tốt nhất')
top_diseases = {}
for label, dset in [('HMDB+CTD', eval_set1), ('SMPDB', eval_set3)]:
    t0 = time.time()
    df_prof = run_loo_eval(dset, run_profancy,  node_idx, N, label=f'PROFANCY/{label}')
    df_cpro = run_loo_eval(dset, _ctqw_pro_fn,  node_idx, N, label=f'CTQW-PRO/{label}')
    merged = df_cpro.merge(df_prof, on='disease', suffixes=('_ctqwpro', '_profancy'))
    merged['delta_mrr'] = merged['mrr_ctqwpro'] - merged['mrr_profancy']
    merged = merged.sort_values('delta_mrr', ascending=False)
    top = merged.head(N_TOP)['disease'].tolist()
    top_diseases[label] = top
    print(f'  {label}: {(time.time()-t0)/60:.1f} min')
    print(f'  Top {N_TOP} bệnh (delta_mrr cao nhất):')
    print(merged.head(N_TOP)[['disease', 'delta_mrr', 'mrr_ctqwpro', 'mrr_profancy']]
          .to_string(index=False))

print('\nSTEP 5 — Chạy dephasing (sigma=0.0 và 5.0) CHỈ trên các bệnh top đã chọn')
dset_by_label = {'HMDB+CTD': eval_set1, 'SMPDB': eval_set3}
results = {}
for label in ['HMDB+CTD', 'SMPDB']:
    full_dset = dset_by_label[label]
    subset = {d: full_dset[d] for d in top_diseases[label] if d in full_dset}
    results[label] = {}
    for sigma in SIGMA_GRID:
        t0 = time.time()
        fn = make_ctqw_pro_dephased(sigma)
        df = run_loo_eval(subset, fn, node_idx, N, label=f'{label} sigma={sigma}')
        results[label][sigma] = df
        if df is not None and not df.empty:
            print(f'  {label} sigma={sigma}: {(time.time()-t0)/60:.1f} min, '
                  f"MRR={df['mrr'].mean():.4f} AUC={df['auc'].mean():.4f} "
                  f"R@20={df['r@20'].mean():.4f}")
        else:
            print(f'  {label} sigma={sigma}: KHÔNG CÓ DATA')

print('\n' + '='*72)
print(f'BẢNG TỔNG HỢP — top {N_TOP} bệnh tốt nhất mỗi bộ, CTQW-PRO (sigma=0) vs '
      f'dephased (sigma=5.0)')
print('  sigma=0.0 PHẢI khớp gần đúng với CTQW-PRO gốc trên đúng tập 15 bệnh này — '
      'sanity check')
print('='*72)
wx_rows = []
for label in ['HMDB+CTD', 'SMPDB']:
    df0 = results[label].get(0.0)
    df5 = results[label].get(5.0)
    print(f'\n--- {label} (n={N_TOP} bệnh top delta_mrr) ---')
    if df0 is not None and not df0.empty:
        print(f"  sigma=0.0: AUC={df0['auc'].mean():.4f} MRR={df0['mrr'].mean():.4f} "
              f"R@5={df0['r@5'].mean():.4f} R@10={df0['r@10'].mean():.4f} "
              f"R@20={df0['r@20'].mean():.4f}")
    if df5 is not None and not df5.empty:
        print(f"  sigma=5.0: AUC={df5['auc'].mean():.4f} MRR={df5['mrr'].mean():.4f} "
              f"R@5={df5['r@5'].mean():.4f} R@10={df5['r@10'].mean():.4f} "
              f"R@20={df5['r@20'].mean():.4f}")
    if df0 is not None and df5 is not None and not df0.empty and not df5.empty:
        df_wx = wilcoxon_table(df5, df0, f'{label} (sigma=5.0 vs sigma=0.0, top-{N_TOP})',
                               method_a='sigma=5.0', method_b='sigma=0.0')
        if df_wx is not None:
            df_wx['dataset'] = label
            wx_rows.append(df_wx)

        # Per-disease breakdown để đọc từng case
        merged2 = df5.merge(df0, on='disease', suffixes=('_s5', '_s0'))
        merged2['delta_mrr_dephase'] = merged2['mrr_s5'] - merged2['mrr_s0']
        print(f'\n  Chi tiết từng bệnh ({label}):')
        print(merged2[['disease', 'mrr_s0', 'mrr_s5', 'delta_mrr_dephase']]
              .sort_values('delta_mrr_dephase')
              .to_string(index=False))

if wx_rows:
    out = pd.concat(wx_rows, ignore_index=True)
    out_path = RESULTS_DIR / 'dephasing_top15_wilcoxon.csv'
    out.to_csv(out_path, index=False)
    print(f'\nSaved: {out_path}')

print('\nDone. Dán TOÀN BỘ output (bảng top-15 mỗi bộ, bảng tổng hợp sigma=0 vs 5.0, '
      'kiểm định, và chi tiết từng bệnh) cho tôi để viết 5.3.')

STEP 1 — Build graph

STEP 2 — Build eval sets (chỉ HMDB+CTD và SMPDB)
  Disease pathways: 20248


SMPDB files:   0%|          | 0/48687 [00:00<?, ?it/s]

  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158 bệnh | SMPDB: 153 bệnh

STEP 3 — Eigendecomposition (Gpro)

STEP 4 — Tính delta_mrr (CTQW-PRO - PROFANCY) trên TOÀN BỘ mỗi bộ dữ liệu, chọn 15 bệnh tốt nhất


PROFANCY/HMDB+CTD:   0%|          | 0/158 [00:00<?, ?it/s]

CTQW-PRO/HMDB+CTD:   0%|          | 0/158 [00:00<?, ?it/s]

...

  HMDB+CTD sigma=5.0: 4.6 min, MRR=0.0481 AUC=0.9081 R@20=0.2253


SMPDB sigma=0.0:   0%|          | 0/15 [00:00<?, ?it/s]

  SMPDB sigma=0.0: 0.5 min, MRR=0.4892 AUC=0.9591 R@20=0.7730


SMPDB sigma=5.0:   0%|          | 0/15 [00:00<?, ?it/s]

  SMPDB sigma=5.0: 1.8 min, MRR=0.1144 AUC=0.9326 R@20=0.2119

BẢNG TỔNG HỢP — top 15 bệnh tốt nhất mỗi bộ, CTQW-PRO (sigma=0) vs dephased (sigma=5.0)
  sigma=0.0 PHẢI khớp gần đúng với CTQW-PRO gốc trên đúng tập 15 bệnh này — sanity check

--- HMDB+CTD (n=15 bệnh top delta_mrr) ---
  sigma=0.0: AUC=0.9462 MRR=0.2970 R@5=0.4425 R@10=0.5579 R@20=0.6176
  sigma=5.0: AUC=0.9081 MRR=0.0481 R@5=0.0455 R@10=0.1153 R@20=0.2253

--- HMDB+CTD (sigma=5.0 vs sigma=0.0, top-15) (n=15) ---
  Metric          sigma=0.0        sigma=5.0    Delta     p_bonf  Sig
  auc        0.9462+/-0.0426   0.9081+/-0.0508  -0.0381  0.0003662  ***
  mrr        0.2970+/-0.0804   0.0481+/-0.0448  -0.2489  0.0003662  ***
  r@5        0.4425+/-0.1393   0.0455+/-0.0593  -0.3970    0.00392  **
  r@10       0.5579+/-0.1765   0.1153+/-0.0954  -0.4426    0.00392  **
  r@20       0.6176+/-0.2159   0.2253+/-0.1538  -0.3923    0.00392  **
  r@50       0.6875+/-0.2007   0.4176+/-0.1741  -0.2699  0.0003662  ***

  Chi tiết từng bệ